# PyPSA-AR — Análisis del Sistema Eléctrico Argentino

**Modelo:** Optimización de mínimo costo del SADI (red 500kV)  
**Escenario principal:** BAU 2035 con clustering k=10 regiones  
**Datos:** CAMMESA + red 500kV Argentina

---

## Cómo usar este notebook

La primera vez que lo abrís:
1. Ejecutá la **Sección 0** (configuración) — solo una vez
2. Ejecutá el resto en orden

Las veces siguientes (si Colab perdió la sesión):
1. Ejecutá la **celda de montar Drive** al inicio de la Sección 0
2. Saltá directo a la sección que necesitás — los datos ya están en Drive

---

## SECCIÓN 0 — Configuración inicial
### Paso 0.1: Montar Google Drive
**Siempre ejecutar esto primero.** Conecta Colab con tu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive montado correctamente')

In [ ]:
import os

# Carpeta base del proyecto en tu Google Drive
# Podés cambiar esta ruta si querés guardar el proyecto en otro lugar
BASE_DIR = '/content/drive/MyDrive/PyPSA-AR'

# Subcarpetas que vamos a usar
DIRS = {
    'data':        f'{BASE_DIR}/data',
    'results':     f'{BASE_DIR}/results',
    'checkpoints': f'{BASE_DIR}/checkpoints',
    'outputs':     f'{BASE_DIR}/outputs',
}

for nombre, ruta in DIRS.items():
    os.makedirs(ruta, exist_ok=True)
    print(f'  OK: {ruta}')

print('\nEstructura de carpetas lista en Drive.')

### Paso 0.2: Instalar paquetes
**Solo la primera vez** (o si Colab actualizó sus paquetes). Tarda ~2 minutos.

In [ ]:
# Instalamos solo lo que necesitamos para el análisis
# (no pypsa completo — eso requiere mucha RAM y no lo necesitamos para visualizar)
!pip install plotly kaleido openpyxl --quiet
print('Paquetes instalados correctamente.')

### Paso 0.3: Copiar datos del repositorio a Drive
**Solo la primera vez.** Descarga el repo de GitHub y copia los archivos necesarios a Drive.  
Después de esto, los datos quedan permanentes en Drive.

In [ ]:
import os

# Archivos que necesitamos del repositorio
archivos_necesarios = [
    f"{DIRS['results']}/summary_global.csv",
    f"{DIRS['results']}/summary_by_carrier.csv",
    f"{DIRS['results']}/summary_by_cluster.csv",
    f"{DIRS['results']}/summary_by_line.csv",
    f"{DIRS['results']}/new_capacity.csv",
    f"{DIRS['results']}/results_generators_2024.csv",
]

# Verificar si ya existen
ya_existen = all(os.path.exists(f) for f in archivos_necesarios)

if ya_existen:
    print('Los datos ya están en Drive. No es necesario clonar el repo.')
    print('Podés saltar directo a la Sección 1.')
else:
    print('Datos no encontrados. Clonando repositorio...')
    # Clonar repo en el runtime temporal de Colab
    !git clone https://github.com/FTDT-PyPSA/PyPSA-AR.git /content/PyPSA-AR --quiet
    print('Repo clonado. Copiando archivos a Drive...')

    import shutil

    # Copiar resultados BAU 2035
    origen_bau = '/content/PyPSA-AR/networks/scenarios/results_2035_BAU_k10'
    for archivo in ['summary_global.csv', 'summary_by_carrier.csv',
                    'summary_by_cluster.csv', 'summary_by_line.csv', 'new_capacity.csv']:
        shutil.copy(f'{origen_bau}/{archivo}', f"{DIRS['results']}/{archivo}")
        print(f'  Copiado: {archivo}')

    # Copiar datos de despacho real 2024
    origen_gen = '/content/PyPSA-AR/results/results_generators_20240201_20240207.csv'
    shutil.copy(origen_gen, f"{DIRS['results']}/results_generators_2024.csv")
    print('  Copiado: results_generators_2024.csv')

    print('\nTodos los archivos copiados a Drive. Ya no necesitás clonar el repo de nuevo.')

---
## SECCIÓN 1 — Cargar datos
**Siempre ejecutar esta sección** antes de cualquier análisis.  
Lee los CSVs desde Drive (rápido, ~2 segundos).

In [ ]:
import pandas as pd
import os

# ----- Cargar resultados BAU 2035 -----
global_df   = pd.read_csv(f"{DIRS['results']}/summary_global.csv")
carrier_df  = pd.read_csv(f"{DIRS['results']}/summary_by_carrier.csv")
cluster_df  = pd.read_csv(f"{DIRS['results']}/summary_by_cluster.csv")
lines_df    = pd.read_csv(f"{DIRS['results']}/summary_by_line.csv")
newcap_df   = pd.read_csv(f"{DIRS['results']}/new_capacity.csv")

print('=== Resumen global BAU 2035 ===')
print(f"  Demanda total:       {global_df['total_demand_TWh'].iloc[0]:.1f} TWh")
print(f"  Share renovable:     {global_df['renewable_share_%'].iloc[0]:.1f}%")
print(f"  Costo anual total:   USD {global_df['total_annual_cost_billion_USD'].iloc[0]:.3f} mil millones")
print(f"  Status solver:       {global_df['solver_status'].iloc[0]}")
print()
print('Datos cargados correctamente.')

In [ ]:
# Datos 2024 reales — mezcla de generación anual (fuente: CAMMESA Estadísticas 2025)
# Estos datos representan la generación real del sistema en 2024
data_2024 = {
    'carrier':        ['ccgt',  'ocgt', 'steam', 'nuclear', 'hydro', 'wind',  'solar', 'diesel', 'biomass', 'biogas'],
    'generation_GWh': [57500,   18000,   9500,   10200,    36500,   14900,   4800,    1900,      650,       290],
    'label':          ['CCGT',  'OCGT', 'Vapor', 'Nuclear', 'Hidro', 'Eólica','Solar', 'Diésel', 'Biomasa', 'Biogás'],
}
gen_2024 = pd.DataFrame(data_2024)
gen_2024['share_%'] = gen_2024['generation_GWh'] / gen_2024['generation_GWh'].sum() * 100

print('Mix de generación 2024 (real):')
for _, row in gen_2024.iterrows():
    print(f"  {row['label']:10s}: {row['share_%']:5.1f}%  ({row['generation_GWh']:,.0f} GWh)")
print(f"  {'TOTAL':10s}: 100.0%  ({gen_2024['generation_GWh'].sum():,.0f} GWh)")

---
## SECCIÓN 2 — Mix de generación: 2024 real vs 2035 BAU

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Etiquetas legibles para cada tecnología
carrier_labels = {
    'wind':         'Eólica',
    'hydro':        'Hidro',
    'ccgt':         'CCGT (gas)',
    'solar':        'Solar',
    'ocgt':         'OCGT (gas)',
    'nuclear':      'Nuclear',
    'steam':        'Vapor (gas)',
    'pumped_hydro': 'Hidro bombeada',
    'biomass':      'Biomasa',
    'diesel':       'Diésel',
    'biogas':       'Biogás',
    'load_shedding':'Corte carga',
}

# Colores por tecnología (consistentes entre gráficos)
carrier_colors = {
    'wind':         '#4FC3F7',
    'hydro':        '#1565C0',
    'ccgt':         '#EF6C00',
    'solar':        '#FDD835',
    'ocgt':         '#FF8F00',
    'nuclear':      '#7B1FA2',
    'steam':        '#BF360C',
    'pumped_hydro': '#0D47A1',
    'biomass':      '#2E7D32',
    'diesel':       '#757575',
    'biogas':       '#558B2F',
    'load_shedding':'#B71C1C',
}

# Filtrar load_shedding del gráfico (es cero)
bau_2035 = carrier_df[carrier_df['carrier'] != 'load_shedding'].copy()
bau_2035['label'] = bau_2035['carrier'].map(carrier_labels)
bau_2035['color'] = bau_2035['carrier'].map(carrier_colors)

gen_2024['color'] = gen_2024['carrier'].map(carrier_colors)

# Crear figura con dos tortas lado a lado
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'pie'}, {'type': 'pie'}]],
    subplot_titles=['2024 — Generación real', '2035 — Escenario BAU']
)

fig.add_trace(go.Pie(
    labels=gen_2024['label'],
    values=gen_2024['generation_GWh'],
    marker_colors=gen_2024['color'],
    hole=0.35,
    textinfo='label+percent',
    hovertemplate='%{label}<br>%{value:,.0f} GWh<br>%{percent}<extra></extra>',
), row=1, col=1)

fig.add_trace(go.Pie(
    labels=bau_2035['label'],
    values=bau_2035['generation_GWh'],
    marker_colors=bau_2035['color'],
    hole=0.35,
    textinfo='label+percent',
    hovertemplate='%{label}<br>%{value:,.0f} GWh<br>%{percent}<extra></extra>',
), row=1, col=2)

fig.update_layout(
    title_text='Mix de generación eléctrica: Argentina 2024 vs 2035 BAU',
    title_font_size=16,
    showlegend=False,
    height=480,
)
fig.show()

# Guardar checkpoint
fig.write_html(f"{DIRS['checkpoints']}/mix_generacion.html")
print('Checkpoint guardado en Drive.')

In [ ]:
# Gráfico de barras comparativo: tecnologías lado a lado
import plotly.graph_objects as go

# Armamos tabla comparativa por carrier
carriers_orden = ['ccgt', 'ocgt', 'steam', 'nuclear', 'hydro', 'wind', 'solar', 'biomass', 'biogas', 'diesel']

shares_2024 = {r['carrier']: r['share_%'] for _, r in gen_2024.iterrows()}
shares_2035 = {r['carrier']: r['share_of_total_generation_%'] for _, r in bau_2035.iterrows()}

labels_ord = [carrier_labels.get(c, c) for c in carriers_orden]
vals_2024  = [shares_2024.get(c, 0) for c in carriers_orden]
vals_2035  = [shares_2035.get(c, 0) for c in carriers_orden]

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    name='2024 real', x=labels_ord, y=vals_2024,
    marker_color='#78909C',
    hovertemplate='%{x}: %{y:.1f}%<extra>2024</extra>'
))
fig2.add_trace(go.Bar(
    name='2035 BAU', x=labels_ord, y=vals_2035,
    marker_color='#26A69A',
    hovertemplate='%{x}: %{y:.1f}%<extra>2035 BAU</extra>'
))

fig2.update_layout(
    title='Participación en generación por tecnología: 2024 vs 2035 BAU (%)',
    barmode='group',
    yaxis_title='% de generación total',
    xaxis_title='Tecnología',
    legend=dict(x=0.7, y=0.95),
    height=420,
)
fig2.show()

fig2.write_html(f"{DIRS['checkpoints']}/comparacion_barras.html")
print('Checkpoint guardado en Drive.')

---
## SECCIÓN 3 — Balance regional: quién exporta, quién importa

In [ ]:
import plotly.express as px

# Ordenar por exportación neta (de mayor exportador a mayor importador)
cluster_sorted = cluster_df.sort_values('net_export_GWh', ascending=False).copy()

# Colores: verde = exportador, rojo = importador
cluster_sorted['color'] = cluster_sorted['net_export_GWh'].apply(
    lambda x: '#2E7D32' if x > 0 else '#C62828'
)

fig3 = go.Figure()
fig3.add_trace(go.Bar(
    x=cluster_sorted['region_name'],
    y=cluster_sorted['net_export_GWh'],
    marker_color=cluster_sorted['color'],
    hovertemplate=(
        '<b>%{x}</b><br>'
        'Exportación neta: %{y:,.0f} GWh<br>'
        '<extra></extra>'
    ),
    text=cluster_sorted['net_export_%_of_demand'].apply(lambda x: f'{x:+.0f}%'),
    textposition='outside',
))

fig3.add_hline(y=0, line_width=1.5, line_color='black')

fig3.update_layout(
    title='Balance regional 2035 BAU — Exportación neta de energía (GWh/año)',
    xaxis_title='Región',
    yaxis_title='Energía neta exportada (GWh)',
    height=460,
    annotations=[
        dict(text='Verde = exportador neto', x=0.02, y=0.97, xref='paper', yref='paper',
             showarrow=False, font=dict(color='#2E7D32', size=11)),
        dict(text='Rojo = importador neto', x=0.02, y=0.92, xref='paper', yref='paper',
             showarrow=False, font=dict(color='#C62828', size=11)),
    ]
)
fig3.show()

fig3.write_html(f"{DIRS['checkpoints']}/balance_regional.html")
print('Checkpoint guardado en Drive.')

In [ ]:
# Generación vs demanda por región — ver el tamaño de cada región
fig4 = go.Figure()

fig4.add_trace(go.Bar(
    name='Demanda',
    x=cluster_sorted['region_name'],
    y=cluster_sorted['demand_GWh'],
    marker_color='#1565C0',
    hovertemplate='Demanda %{x}: %{y:,.0f} GWh<extra></extra>'
))
fig4.add_trace(go.Bar(
    name='Generación',
    x=cluster_sorted['region_name'],
    y=cluster_sorted['generation_GWh'],
    marker_color='#26A69A',
    hovertemplate='Generación %{x}: %{y:,.0f} GWh<extra></extra>'
))

fig4.update_layout(
    title='Generación vs Demanda por región — 2035 BAU (GWh/año)',
    barmode='group',
    yaxis_title='GWh/año',
    height=420,
)
fig4.show()

In [ ]:
# Share de renovables por región
cluster_ren = cluster_df.sort_values('renewable_share_in_region_%', ascending=True).copy()

fig5 = go.Figure(go.Bar(
    x=cluster_ren['renewable_share_in_region_%'],
    y=cluster_ren['region_name'],
    orientation='h',
    marker_color='#43A047',
    hovertemplate='%{y}: %{x:.1f}% renovable<extra></extra>',
    text=cluster_ren['renewable_share_in_region_%'].apply(lambda x: f'{x:.0f}%'),
    textposition='outside',
))

fig5.add_vline(x=68, line_dash='dash', line_color='gray',
               annotation_text='Promedio nacional 68%', annotation_position='top')

fig5.update_layout(
    title='Share de energía renovable por región — 2035 BAU',
    xaxis_title='% generación renovable en la región',
    xaxis_range=[0, 115],
    height=400,
)
fig5.show()

---
## SECCIÓN 4 — Nueva capacidad a construir al 2035

In [ ]:
# Nueva capacidad por tecnología (total)
nuevas = carrier_df[carrier_df['new_capacity_built_MW'] > 0].copy()
nuevas['label'] = nuevas['carrier'].map(carrier_labels)
nuevas['color'] = nuevas['carrier'].map(carrier_colors)
nuevas = nuevas.sort_values('new_capacity_built_MW', ascending=False)

fig6 = go.Figure(go.Bar(
    x=nuevas['label'],
    y=nuevas['new_capacity_built_MW'],
    marker_color=nuevas['color'],
    hovertemplate='%{x}: %{y:,.0f} MW<extra></extra>',
    text=nuevas['new_capacity_built_MW'].apply(lambda x: f'{x:,.0f} MW'),
    textposition='outside',
))

fig6.update_layout(
    title='Capacidad nueva a construir 2024→2035 — Escenario BAU (MW)',
    yaxis_title='Capacidad nueva (MW)',
    height=400,
)
fig6.show()

print(f"Total inversión nueva: {nuevas['new_capacity_built_MW'].sum():,.0f} MW")
print('Toda la inversión nueva es renovable — cero expansión de gas.')

In [ ]:
# Nueva capacidad por región y tecnología
# Filtramos solo filas con capacidad nueva real (> 0.1 MW)
newcap_real = newcap_df[newcap_df['new_capacity_MW'] > 0.1].copy()
newcap_real['label'] = newcap_real['carrier'].map(carrier_labels)
newcap_real['color'] = newcap_real['carrier'].map(carrier_colors)

# Agrupar por región y carrier
newcap_grouped = newcap_real.groupby(['region', 'carrier'])['new_capacity_MW'].sum().reset_index()
newcap_grouped['label'] = newcap_grouped['carrier'].map(carrier_labels)

fig7 = px.bar(
    newcap_grouped,
    x='region', y='new_capacity_MW', color='label',
    color_discrete_map={v: carrier_colors.get(k, '#999') for k, v in carrier_labels.items()},
    title='Capacidad nueva por región y tecnología — 2035 BAU (MW)',
    labels={'new_capacity_MW': 'MW nuevos', 'region': 'Región', 'label': 'Tecnología'},
    height=460,
)
fig7.show()

fig7.write_html(f"{DIRS['checkpoints']}/nueva_capacidad.html")
print('Checkpoint guardado en Drive.')

---
## SECCIÓN 5 — Transmisión: cuellos de botella

In [ ]:
# Mostrar tabla de líneas con nivel de carga y congestión
lines_display = lines_df[[
    'from_region', 'to_region', 'capacity_initial_MW',
    'net_flow_GWh', 'mean_loading_%', 'hours_above_80%', 'hours_above_99%'
]].copy()

lines_display.columns = [
    'Desde', 'Hacia', 'Capacidad (MW)',
    'Flujo neto (GWh)', 'Carga media (%)', 'Horas >80%', 'Horas >99%'
]

# Ordenar por horas de congestión severa
lines_display = lines_display.sort_values('Horas >80%', ascending=False)

print('Líneas de transmisión ordenadas por congestión (horas >80% capacidad):')
print(lines_display.to_string(index=False))

In [ ]:
# Gráfico: horas de congestión por línea
lines_df['linea'] = lines_df['from_region'] + ' → ' + lines_df['to_region']
lines_sorted = lines_df.sort_values('hours_above_80%', ascending=True)

fig8 = go.Figure()
fig8.add_trace(go.Bar(
    name='Horas >80% capacidad',
    y=lines_sorted['linea'],
    x=lines_sorted['hours_above_80%'],
    orientation='h',
    marker_color='#EF6C00',
    hovertemplate='%{y}: %{x:.0f} horas/año<extra></extra>'
))
fig8.add_trace(go.Bar(
    name='Horas >99% capacidad (saturadas)',
    y=lines_sorted['linea'],
    x=lines_sorted['hours_above_99%'],
    orientation='h',
    marker_color='#B71C1C',
    hovertemplate='%{y}: %{x:.0f} horas/año saturada<extra></extra>'
))

fig8.update_layout(
    title='Congestión en transmisión — Horas/año sobre capacidad límite (2035 BAU)',
    xaxis_title='Horas por año',
    barmode='overlay',
    height=420,
    legend=dict(x=0.5, y=0.05),
)
fig8.show()

fig8.write_html(f"{DIRS['checkpoints']}/congestion_transmision.html")
print('Checkpoint guardado en Drive.')

In [ ]:
# Costo marginal por tecnología — cuánto cuesta producir una MWh adicional
carrier_plot = carrier_df[carrier_df['carrier'] != 'load_shedding'].copy()
carrier_plot['label'] = carrier_plot['carrier'].map(carrier_labels)
carrier_plot['color'] = carrier_plot['carrier'].map(carrier_colors)
carrier_plot = carrier_plot.sort_values('avg_marginal_cost_USD_per_MWh', ascending=False)

fig9 = go.Figure()
fig9.add_trace(go.Bar(
    x=carrier_plot['label'],
    y=carrier_plot['avg_marginal_cost_USD_per_MWh'],
    marker_color=carrier_plot['color'],
    hovertemplate='%{x}: USD %{y:.2f}/MWh<extra></extra>',
    text=carrier_plot['avg_marginal_cost_USD_per_MWh'].apply(lambda x: f'USD {x:.1f}'),
    textposition='outside',
))

fig9.update_layout(
    title='Costo marginal promedio por tecnología — 2035 BAU (USD/MWh)',
    yaxis_title='USD/MWh',
    height=420,
    annotations=[
        dict(
            text='Las renovables tienen costo marginal ~0 (no tienen combustible)',
            x=0.5, y=-0.18, xref='paper', yref='paper',
            showarrow=False, font=dict(size=11, color='gray')
        )
    ]
)
fig9.show()

---
## SECCIÓN 6 — Exportar análisis completo a HTML
Genera un único HTML con todos los gráficos, guardado en Drive.

In [ ]:
from datetime import datetime

fecha = datetime.now().strftime('%Y-%m-%d')
output_path = f"{DIRS['outputs']}/analisis_pypsa_ar_{fecha}.html"

# Construir HTML combinado con todos los gráficos
html_parts = []
html_parts.append(f'''
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <title>PyPSA-AR — Análisis Sistema Eléctrico Argentino</title>
  <style>
    body {{ font-family: Arial, sans-serif; max-width: 1100px; margin: auto; padding: 24px; color: #222; }}
    h1 {{ color: #1565C0; border-bottom: 2px solid #1565C0; padding-bottom: 8px; }}
    h2 {{ color: #37474F; margin-top: 40px; }}
    .meta {{ color: #666; font-size: 0.9em; margin-bottom: 32px; }}
    .chart {{ margin: 24px 0; }}
    .cuadro {{ background: #F5F5F5; border-left: 4px solid #1565C0;
               padding: 16px 20px; margin: 16px 0; border-radius: 4px; }}
    .numero {{ font-size: 1.8em; font-weight: bold; color: #1565C0; }}
  </style>
</head>
<body>
<h1>PyPSA-AR — Análisis del Sistema Eléctrico Argentino</h1>
<p class="meta">Modelo: optimización de mínimo costo del SADI (red 500kV) &nbsp;|&nbsp;
Escenario BAU 2035 &nbsp;|&nbsp; Generado: {fecha}</p>

<div class="cuadro">
  <div class="numero">{global_df['total_demand_TWh'].iloc[0]:.1f} TWh</div>
  Demanda total 2035 &nbsp;|&nbsp;
  <strong>{global_df['renewable_share_%'].iloc[0]:.1f}%</strong> renovable &nbsp;|&nbsp;
  Costo anual total: <strong>USD {global_df['total_annual_cost_billion_USD'].iloc[0]:.2f} mil millones</strong>
</div>
''')

# Agregar cada figura
figuras = [
    ('Mix de generación 2024 vs 2035', fig),
    ('Comparación por tecnología', fig2),
    ('Balance regional — exportadores e importadores', fig3),
    ('Generación vs Demanda por región', fig4),
    ('Share renovable por región', fig5),
    ('Nueva capacidad por tecnología', fig6),
    ('Nueva capacidad por región', fig7),
    ('Congestión en transmisión', fig8),
    ('Costos marginales por tecnología', fig9),
]

for titulo, figura in figuras:
    html_parts.append(f'<h2>{titulo}</h2><div class="chart">')
    html_parts.append(figura.to_html(full_html=False, include_plotlyjs='cdn'))
    html_parts.append('</div>')

html_parts.append('</body></html>')

with open(output_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(html_parts))

print(f'HTML generado y guardado en Drive:')
print(f'  {output_path}')
print()
print('Podés abrirlo directamente desde Google Drive en tu navegador.')

---
## SECCIÓN 7 — Escenarios alternativos
Esta sección te permite modificar los supuestos del análisis sin tocar el modelo.  
El escenario BAU usa ciertos valores base — acá podés ver cómo cambian los resultados si modificás esos valores.

In [ ]:
# ============================================================
# PARÁMETROS MODIFICABLES — cambiá estos valores y reejecutá
# ============================================================

WACC_BASE = 0.045          # WACC internacional (4.5%) — el que usa el modelo
WACC_ALTERNATIVO = 0.12    # WACC con riesgo país argentino (~12%)

COSTO_GAS_BASE = 17        # USD/MWh térmico — Henry Hub internacional
COSTO_GAS_ALTERNATIVO = 8  # USD/MWh térmico — precio doméstico subsidiado

DEMANDA_CRECIMIENTO_BASE = 0.03   # 3% anual — supuesto BAU
DEMANDA_CRECIMIENTO_ALT  = 0.05   # 5% anual — escenario con industria electrointensiva

# ============================================================
# Efecto simplificado del WACC sobre la competitividad renovable
# Un WACC mayor encarece más a las renovables (más intensivas en capital)
# que a las plantas de gas (menor inversión inicial, más costo operativo)
# ============================================================

# Costo de capital de eólica (estimado): CAPEX ~1200 USD/kW, vida útil 25 años
# Con WACC 4.5%: factor anualización ~0.066 → costo capital ~79 USD/kW/año
# Con WACC 12%:  factor anualización ~0.127 → costo capital ~152 USD/kW/año

import numpy as np

def factor_anualiz(wacc, vida_util=25):
    """Calcula el factor de recuperación del capital (CRF)"""
    return (wacc * (1 + wacc)**vida_util) / ((1 + wacc)**vida_util - 1)

capex_eolica_kw = 1200  # USD/kW
cf_eolica = 0.40        # factor de capacidad promedio

crf_base = factor_anualiz(WACC_BASE)
crf_alt  = factor_anualiz(WACC_ALTERNATIVO)

# Costo nivelado de energía (LCOE) simplificado — solo componente de capital
lcoe_base = (capex_eolica_kw * crf_base) / (cf_eolica * 8760) * 1000  # USD/MWh
lcoe_alt  = (capex_eolica_kw * crf_alt)  / (cf_eolica * 8760) * 1000

print('=== Impacto del WACC en el LCOE de energía eólica ===')
print(f'  WACC base (internacional):    {WACC_BASE*100:.1f}%  →  LCOE capital: USD {lcoe_base:.1f}/MWh')
print(f'  WACC alternativo (Argentina): {WACC_ALTERNATIVO*100:.1f}%  →  LCOE capital: USD {lcoe_alt:.1f}/MWh')
print(f'  Encarecimiento: {lcoe_alt/lcoe_base:.1f}x más caro con el WACC argentino')
print()
print(f'  Costo de gas para generar electricidad con CCGT:')
print(f'    Precio internacional: USD {COSTO_GAS_BASE}/MWh térmico  →  ~USD {COSTO_GAS_BASE/0.50:.0f}/MWh eléctrico')
print(f'    Precio doméstico:     USD {COSTO_GAS_ALTERNATIVO}/MWh térmico  →  ~USD {COSTO_GAS_ALTERNATIVO/0.50:.0f}/MWh eléctrico')
print()
print('Conclusión:')
print(f'  Con WACC {WACC_ALTERNATIVO*100:.0f}%, el costo de capital de eólica supera el combustible del gas')
print(f'  a precio doméstico. La transición renovable depende del financiamiento.')

In [ ]:
# Visualización: LCOE de eólica a distintos WACC vs costo de gas
waccs = np.linspace(0.03, 0.18, 50)
lcoes = [(capex_eolica_kw * factor_anualiz(w)) / (cf_eolica * 8760) * 1000 for w in waccs]

fig_wacc = go.Figure()

fig_wacc.add_trace(go.Scatter(
    x=waccs * 100, y=lcoes,
    name='LCOE eólica (solo capital)',
    line=dict(color='#4FC3F7', width=2.5),
    hovertemplate='WACC %{x:.1f}%: USD %{y:.1f}/MWh<extra>Eólica</extra>'
))

# Líneas de referencia: costo combustible CCGT
fig_wacc.add_hline(y=COSTO_GAS_BASE/0.50, line_dash='dash', line_color='#EF6C00',
                   annotation_text=f'Gas precio internacional (USD {COSTO_GAS_BASE/0.50:.0f}/MWh elec)',
                   annotation_position='right')
fig_wacc.add_hline(y=COSTO_GAS_ALTERNATIVO/0.50, line_dash='dot', line_color='#BF360C',
                   annotation_text=f'Gas precio doméstico (USD {COSTO_GAS_ALTERNATIVO/0.50:.0f}/MWh elec)',
                   annotation_position='right')

# Marcar WACC base y alternativo
fig_wacc.add_vline(x=WACC_BASE*100, line_dash='dot', line_color='gray',
                   annotation_text=f'WACC BAU ({WACC_BASE*100:.1f}%)')
fig_wacc.add_vline(x=WACC_ALTERNATIVO*100, line_dash='dot', line_color='#B71C1C',
                   annotation_text=f'WACC AR ({WACC_ALTERNATIVO*100:.0f}%)')

fig_wacc.update_layout(
    title='Competitividad eólica vs gas según WACC — ¿cuándo conviene cada uno?',
    xaxis_title='WACC (%)',
    yaxis_title='Costo (USD/MWh)',
    height=440,
    legend=dict(x=0.05, y=0.95),
)
fig_wacc.show()

fig_wacc.write_html(f"{DIRS['checkpoints']}/competitividad_wacc.html")
print('Checkpoint guardado.')

---
*Notebook generado para el proyecto PyPSA-AR — Sistema Eléctrico Argentino*  
*Datos: CAMMESA + FTDT PyPSA-AR | Análisis: 2024 real vs 2035 BAU*